In [ ]:
!pip install datasets
!pip install evaluate
!git clone https://github.com/LyzJames/graph.git
%cd graph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 9.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.0 MB/s eta 0:00:00
Cloning into 'graph'...
remote: Enumerating objects: 61, done.
remote: Counting obje

In [ ]:
import pandas as pd
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import evaluate
import logging
import graph
from google.colab import drive
from transformers import DataCollatorWithPadding, create_optimizer
from transformers import TFAutoModelForSequenceClassification, DistilBertConfig
from transformers.keras_callbacks import KerasMetricCallback
from transformers import AutoTokenizer
from datasets import load_dataset, Dataset

In [ ]:
# Load dataset
ds = load_dataset("dair-ai/emotion", "unsplit")

data = ds['train'].to_pandas()

# Print the shape of the DataFrame
print("Data shape:", data.shape)

# Print the first few rows to check
print("\nData head:")
print(data.head())

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/26.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/416809 [00:00<?, ? examples/s]

Data shape: (416809, 2)

Data head:
                                                text  label
0  i feel awful about it too because it s my job ...      0
1                              im alone i feel awful      0
2  ive probably mentioned this before but i reall...      1
3           i was feeling a little low few days back      0
4  i beleive that i am much more sensitive to oth...      2


In [ ]:
df_list = []
db = data.copy()

for num in [3000,1000,6000]:
  # Split the data based on label values
  sadness_data = db[db['label'] == 0].iloc[:num]
  joy_data = db[db['label'] == 1].iloc[:num]
  love_data = db[db['label'] == 2].iloc[:num]
  anger_data = db[db['label'] == 3].iloc[:num]
  fear_data = db[db['label'] == 4].iloc[:num]
  surprise_data = db[db['label'] == 5].iloc[:num]

  # Combine the data into a single DataFrame
  df = pd.concat([sadness_data, joy_data, love_data, anger_data, fear_data, surprise_data])

  # Remove the sampled data from the original dataset
  db = db.drop(df.index)

  df_list.append(df)

df_test = df_list[0].sample(frac=1, random_state=42).reset_index(drop=True)
df_validation = df_list[1].sample(frac=1, random_state=42).reset_index(drop=True)
df_train = df_list[2].sample(frac=1, random_state=42).reset_index(drop=True)

# Print shapes
print("Train data shape:", df_train.shape)
print("Test data shape:", df_test.shape)
print("Validation data shape:", df_validation.shape)

# Print the first rows for each dataset to check
print("\nTrain data head:")
print(df_train.head())

print("\nTest data head:")
print(df_test.head())

print("\nValidation data head:")
print(df_validation.head())

Train data shape: (36000, 2)
Test data shape: (18000, 2)
Validation data shape: (6000, 2)

Train data head:
                                                text  label
0   i also feel strangely affectionate towards frank      2
1  i feel more impatient about giving birth this ...      3
2  ive always loved very but i kind of feel like ...      3
3                       i feel completely indecisive      4
4  i feel very virtuous from having exercised tha...      1

Test data head:
                                                text  label
0  i was feeling homesick i would sneak up to the...      0
1  i feel sympathetic for johnny because he is di...      2
2  i dont believe the past year has been time was...      3
3  i opted for drugs to alleviate the pain but i ...      1
4  id feel more embarrassed that i already do by ...      0

Validation data head:
                                                text  label
0  i couldnt possibly tell you about any of them ...      1
1  i dunno i

In [ ]:
df_train, df_validation, df_test = graph.Graph_Based_Processing(df_train, df_validation, df_test)

# Print the first rows for each dataset to check
print("\nGraph Train data head:")
print(df_train.head())

print("\nGraph Test data head:")
print(df_test.head())

print("\nGraph Validation data head:")
print(df_validation.head())

Processing texts: 100%|██████████| 60000/60000 [00:04<00:00, 14093.43text/s]
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.



Graph Train data head:
                                                text  label
0  (also) (feel) (strangely) (affectionate) (towa...      2
1        (ifeel) (impatient) (giving) (birth) (time)      3
2  (ive) (always) (loved) (kind) (feellike) (they...      3
3                  (ifeel) (completely) (indecisive)      4
4              (ifeel) (virtuous) (exercised) (much)      1

Graph Test data head:
                                                text  label
0  (feeling) (homesick) (would) (sneak) (storage)...      0
1  (ifeel) (sympathetic) (johnny) (different) (re...      2
2  (dont) (believe) (past) (year) (time) (wasted)...      3
3  (opted) (drugs) (alleviate) (pain) (lucky) (en...      1
4  (id) (feel) (embarrassed) (already) (pointing)...      0

Graph Validation data head:
                                                text  label
0  (couldnt) (possibly) (tell) (without) (feeling...      1
1      (dunno) (feel) (insulted) (take) (compliment)      3
2        (amfeeling) (re

In [ ]:
def preprocess_text(text):
  return text.replace("(", "").replace(")", "")

df_train['text'] = df_train['text'].apply(preprocess_text)
df_test['text'] = df_test['text'].apply(preprocess_text)
df_validation['text'] = df_validation['text'].apply(preprocess_text)

train_data = Dataset.from_pandas(df_train, preserve_index=False)
test_data = Dataset.from_pandas(df_test, preserve_index=False)
validation_data = Dataset.from_pandas(df_validation, preserve_index=False)

In [ ]:
# Initialize the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

def preprocess_function(db):
    return tokenizer(db['text'], truncation=True)

tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)
tokenized_validation = validation_data.map(preprocess_function, batched=True)

# Create a data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="tf")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/36000 [00:00<?, ? examples/s]

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

In [ ]:
# Mapping from ID to label
id2label = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

# Mapping from label to ID
label2id = {
    "sadness": 0,
    "joy": 1,
    "love": 2,
    "anger": 3,
    "fear": 4,
    "surprise": 5
}

batch_size = 32
num_epochs = 5
batches_per_epoch = len(tokenized_train) // batch_size
total_train_steps = int(batches_per_epoch * num_epochs)
optimizer, schedule = create_optimizer(init_lr=0.00017, num_warmup_steps=0, num_train_steps=total_train_steps)

config = DistilBertConfig(num_labels=6, id2label=id2label, label2id=label2id)
DistilBERT_model = TFAutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased", config=config)
DistilBERT_model.compile(optimizer=optimizer, metrics=['accuracy'])

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_transform.weight', 'vocab_transform.bias', 'vocab_layer_norm.weight']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

In [ ]:
tf_train_set = DistilBERT_model.prepare_tf_dataset(
    tokenized_train,
    shuffle=True,
    batch_size=batch_size,
    collate_fn=data_collator,
)

tf_validation_set = DistilBERT_model.prepare_tf_dataset(
    tokenized_validation,
    shuffle=False,
    batch_size=batch_size,
    collate_fn=data_collator,
)

tf_test_set = DistilBERT_model.prepare_tf_dataset(
    tokenized_test,
    shuffle=False,
    batch_size=batch_size,
    collate_fn=data_collator,
)

In [ ]:
print(DistilBERT_model.summary())

Model: "tf_distil_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 distilbert (TFDistilBertMa  multiple                  66362880  
 inLayer)                                                        
                                                                 
 pre_classifier (Dense)      multiple                  590592    
                                                                 
 classifier (Dense)          multiple                  4614      
                                                                 
 dropout_19 (Dropout)        multiple                  0 (unused)
                                                                 
Total params: 66958086 (255.42 MB)
Trainable params: 66958086 (255.42 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [ ]:
f1 = evaluate.load("f1")
precision = evaluate.load("precision")
recall = evaluate.load("recall")

def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  predictions = np.argmax(predictions, axis=1)

  f1_score = f1.compute(predictions=predictions, references=labels, average="weighted")
  precision_score = precision.compute(predictions=predictions, references=labels, average="weighted")
  recall_score = recall.compute(predictions=predictions, references=labels, average="weighted")

  return {
      "f1": f1_score["f1"],
      "precision": precision_score["precision"],
      "recall": recall_score["recall"],
      }

metric_callback = KerasMetricCallback(metric_fn=compute_metrics, eval_dataset=tf_validation_set)
callbacks = [metric_callback]

In [ ]:
DistilBERT_model.fit(tf_train_set, validation_data=tf_validation_set, epochs=num_epochs, callbacks=callbacks)

Epoch 1/5
1125/1125 [==============================] - 165s 126ms/step - loss: 0.3287 - accuracy: 0.8878 - val_loss: 0.1516 - val_accuracy: 0.9452 - f1: 0.9449 - precision: 0.9478 - recall: 0.9452
Epoch 2/5
1125/1125 [==============================] - 135s 120ms/step - loss: 0.1450 - accuracy: 0.9449 - val_loss: 0.1451 - val_accuracy: 0.9478 - f1: 0.9475 - precision: 0.9509 - recall: 0.9478
Epoch 3/5
1125/1125 [==============================] - 134s 120ms/step - loss: 0.1185 - accuracy: 0.9516 - val_loss: 0.1331 - val_accuracy: 0.9495 - f1: 0.9493 - precision: 0.9509 - recall: 0.9495
Epoch 4/5
1125/1125 [==============================] - 134s 119ms/step - loss: 0.0982 - accuracy: 0.9579 - val_loss: 0.1380 - val_accuracy: 0.9487 - f1: 0.9484 - precision: 0.9501 - recall: 0.9487
Epoch 5/5
1125/1125 [==============================] - 134s 119ms/step - loss: 0.0801 - accuracy: 0.9651 - val_loss: 0.1522 - val_accuracy: 0.9480 - f1: 0.9478 - precision: 0.9492 - recall: 0.9480


In [ ]:
# Evaluate the model
loss, accuracy = DistilBERT_model.evaluate(tf_test_set)
print(f"Test Accuracy: {accuracy}")

563/563 [==============================] - 21s 38ms/step - loss: 0.1545 - accuracy: 0.9471
Test Accuracy: 0.9471111297607422


In [ ]:
drive.mount('/content/drive')
DistilBERT_model.save_pretrained('/content/drive/MyDrive/DistilBERT_With_Graph_Data_model')

Mounted at /content/drive
